# 02. FinBERT zero-shot baseline

Runs `ProsusAI/finbert` as-is on the labeled FOMC sentences, with no training. FinBERT was trained for general financial sentiment (positive / negative / neutral), not hawkish / dovish stance, so we expect weak results. This is the floor the fine-tuned models in notebook 03 should beat.

Mapping used: positive -> hawkish, negative -> dovish, neutral -> neutral. Zero-shot never sees the labels, so every kept sentence is scored once and we report on the same test sets the trained models use (time split, and minutes only).

Runtime: GPU (Runtime > Change runtime type > T4).

In [ ]:
# Colab: clone the repo and rebuild the processed data (raw and processed data are not in git)
import os
if not os.path.exists("DSA495-Fed-Reader") and not os.path.exists("src"):
    !git clone https://github.com/kgpai97-lab/DSA495-Fed-Reader.git
if os.path.exists("DSA495-Fed-Reader"):
    %cd DSA495-Fed-Reader
!git pull
!pip install -q -r requirements.txt
!python src/load_fomc_dataset.py
!python src/recover_metadata.py
!python src/flag_sentences.py
!python src/prepare_splits.py

In [ ]:
import sys
sys.path.insert(0, "src")
import numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tone_eval import load_splits, score_table, LABELS, RESULTS_DIR

df = load_splits()
device = "cuda" if torch.cuda.is_available() else "cpu"
print(len(df), "sentences;", device)

In [ ]:
MODEL_NAME = "ProsusAI/finbert"
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device).eval()
print(model.config.id2label)

# FinBERT label -> our label id (0 dovish, 1 hawkish, 2 neutral)
TO_OURS = {"negative": 0, "positive": 1, "neutral": 2}
col_for_ours = [next(i for i, n in model.config.id2label.items() if TO_OURS[n.lower()] == k) for k in range(3)]

In [ ]:
@torch.no_grad()
def predict_probs(sentences, batch_size=64):
    out = []
    for i in range(0, len(sentences), batch_size):
        enc = tok(sentences[i:i + batch_size], truncation=True, max_length=128, padding=True, return_tensors="pt").to(device)
        out.append(torch.softmax(model(**enc).logits, dim=-1).cpu().numpy())
    return np.vstack(out)[:, col_for_ours]  # columns now ordered dovish, hawkish, neutral

probs = predict_probs(df["sentence"].tolist())
pred_df = df[["label", "doc_type", "year", "time_split", "official_split"]].copy()
pred_df[["p_dovish", "p_hawkish", "p_neutral"]] = probs
pred_df["pred"] = probs.argmax(1)

## Scores
The time-split test set (years after 2018) is the headline number. The all-sentences row is shown for reference; zero-shot has no training leakage to worry about.

In [ ]:
results = pd.concat([
    score_table(pred_df[pred_df["time_split"] == "test"], "FinBERT zero-shot (time-split test)"),
    score_table(pred_df, "FinBERT zero-shot (all kept sentences)"),
], ignore_index=True)
results.round(3)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

test = pred_df[pred_df["time_split"] == "test"]
print(classification_report(test["label"], test["pred"], target_names=LABELS, zero_division=0))
ConfusionMatrixDisplay.from_predictions(test["label"], test["pred"], display_labels=LABELS, cmap="Blues")
plt.title("FinBERT zero-shot, time-split test")
plt.tight_layout(); plt.show()

In [ ]:
# Confident mistakes, for the error discussion
wrong = pred_df[pred_df["pred"] != pred_df["label"]].copy()
wrong["conf"] = wrong[["p_dovish", "p_hawkish", "p_neutral"]].max(axis=1)
wrong["sentence"] = df.loc[wrong.index, "sentence"]
for _, r in wrong.sort_values("conf", ascending=False).head(8).iterrows():
    print(f"true={LABELS[r.label]:8s} pred={LABELS[r.pred]:8s} conf={r.conf:.2f}\n  {r.sentence}\n")

In [ ]:
RESULTS_DIR.mkdir(exist_ok=True)
pred_df.to_csv(RESULTS_DIR / "preds_finbert_zeroshot.csv")
results.to_csv(RESULTS_DIR / "scores_finbert_zeroshot.csv", index=False)
print("saved to results/ (download from the Colab file browser and commit if wanted)")